In [ ]:
import os
import sys
sys.path.append(os.path.dirname(os.getcwd()))

import time
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm
from matplotlib.colors import Normalize
from PIL import Image

from pc import PS
from modules import ADC,DAC,CHIP,SELECT
from command import CMD,CmdData,Packet
from command.singleCmdInfo import *

from util import plot_v_cond,plot_cond,show_crossbar
from numpy import printoptions
import pickle
from scipy.stats import norm
from scipy.optimize import curve_fit

In [ ]:
chip=CHIP(PS(host="192.168.1.11", port = 7, debug=0),init=True)
chip.set_device_cfg(deviceType=0,IsNew32=True)
chip.adc.set_gap(adc_cs_gap=100,adc_first_gap=20,adc_last_gap=10)
chip.adc.set_gain_resistor(big_resistance=10e3,small_resistance=200)
chip.adc.set_sample_times(adc_sample_times=32)
chip.clk_manager.set_cyc(10, 10,delay3=100)
chip.add_compiler("../compiler/code/")

In [ ]:
crossbar = np.ones((256,256))
v,c,r = chip.read4(crossbar=crossbar,row_index=None,col_index=None,read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=False,split_type=0,row_type=0,col_type=0)
plot_cond(c,vmax=1000)

In [ ]:
plot_cond(c,vmax=1000)

In [ ]:
chip.adc.set_gap(adc_cs_gap=100,adc_first_gap=1000,adc_last_gap=10)

In [ ]:
crossbar = np.ones((256,256))
ans = []
ans1 = []
row =11
col =100
for i in range(10):
    # time.sleep(4)
    v,cond,_ = chip.read4(crossbar=crossbar,row_index=[row],col_index=[col],read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=True,split_type=7,row_type=0,col_type=0)
    # _,_,_ = chip.read4(crossbar=crossbar,row_index=[row+1],col_index=[col+1],read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=True,split_type=7,row_type=0,col_type=0)
    ans.append(cond[row,col])
    ans1.append(v[row,col])
# plt.plot(ans[:])
# plt.show()
# plt.plot(ans1[:])
# plt.show()
# time.sleep(3)
# for i in range(20):
#     v,cond,_ = chip.read4(crossbar=crossbar,row_index=[row],col_index=[col],read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=True,split_type=7,row_type=0,col_type=0)
#     _,_,_ = chip.read4(crossbar=crossbar,row_index=[row+1],col_index=[col+1],read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=True,split_type=7,row_type=0,col_type=0)
#     ans.append(cond[row,col])
#     ans1.append(v[row,col])
# # plot_cond(cond)
# plt.plot(ans[:])
# plt.show()
# plt.plot(ans1[:])
# plt.show()

In [ ]:
plt.plot(ans[:])
plt.show()
plt.plot(ans1[:])
plt.show()

In [ ]:
print(np.mean(ans1))
print(np.std(ans1))
print(np.std(ans1)/np.mean(ans1))

In [ ]:
# set_good_device_file_name = "../data/good_device/cond_500_r8.npy"
# reset_good_device_file_name = "../data/good_device/cond_200_r8.npy"

In [ ]:
select = SELECT()

### 1. Reset操作

In [ ]:
for i in range(1):
      select.Reset(chip=chip,need_read = np.ones((256,256),dtype=bool),write_times=61,start_v=2,delta_v=0.05,tg=5,threshold=200,
            reset_pulse_width=1e-3,sub_base=False,plot_cond=plot_cond,vmax=1400)

### Reset stuck-on fault devices

In [ ]:
chip.ps.set_time_out(10)
for i in range(10):
      select.Reset(chip=chip,need_read = np.ones((256,256),dtype=bool),write_times=61,start_v=2,delta_v=0.05,tg=5,threshold=200,
            reset_pulse_width=10e-3,sub_base=False,plot_cond=plot_cond,vmax=1400)

In [ ]:
crossbar = np.ones((256,256))
_,cond,_ = chip.read4(crossbar=crossbar,row_index=None,col_index=None,read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=True,split_type=0,row_type=0,col_type=0)
plot_cond(cond)
print(np.sum(cond<200))
# reset_good_device_mask = cond<200
reset_good_device_mask2 = cond<200

### Set (for good devices)

In [ ]:
need_read = np.ones((256,256),dtype=bool)
select.Set(chip=chip,need_read=need_read,write_times=41,write_voltage=5,start_tg=1,delta_tg=0.05,threshold=800,set_pulse_width=100e-6,
           sub_base=True,vmax=1000,plot_cond=plot_cond)

In [ ]:
crossbar = np.ones((256,256))
_,cond,_ = chip.read4(crossbar=crossbar,row_index=None,col_index=None,read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=True,split_type=0,row_type=0,col_type=0)
plot_cond(cond)
print(np.sum(cond>800))
# set_good_device_mask = cond>800
# set_good_device_mask2 = cond>800

In [ ]:
good_device_mask2 = reset_good_device_mask2 & set_good_device_mask2
print(np.sum(good_device_mask2))
np.save("../data/good_device/20250621_cond_reset_200_set_500.npy", good_device_mask2)

In [ ]:
def create_tg_cond_mask(arr, t, tg, plot_mask, input_mask=None):
    # 如果没有提供input_mask，则创建全True掩码
    if input_mask is None:
        input_mask = np.ones_like(arr, dtype=bool)
    masked_arr = arr[input_mask]
    data = masked_arr.flatten()

    mu, sigma = norm.fit(data)
    # 计算有效范围
    lower_bound = mu - t
    upper_bound = mu + t
    # 计算范围并生成掩码（仅当值在范围内且input_mask为True时返回True）
    range_mask = (arr >= lower_bound) & (arr <= upper_bound)
    return_mask = range_mask & input_mask

    plt.figure(figsize=(10, 6))
    plt.hist(arr[plot_mask].flatten(), bins=50, density=True, alpha=0.5, color='b', label='actual dist')
    # 绘制拟合曲线
    x = np.linspace(min(data), max(data), 100)
    pdf = norm.pdf(x, mu, sigma)  # 计算概率密度
    plt.plot(x, pdf, 'r-', linewidth=2, label='fit dist')
    plt.title(f"tg_{tg}: mu={np.mean(data):.2f}, sigma={np.var(data):.2f}\nfit_mu1={mu}, fit_sigma1={sigma}")
    plt.xlabel("Conductance (uS)")
    plt.ylabel("Frequency")
    plt.legend()
    plt.show()
    # with printoptions(threshold=np.inf):  # 仅在此上下文生效
    #     print(counts)
    
    return mu, return_mask

In [ ]:
def Set(chip,need_read,write_times,write_voltage,start_tg,delta_tg,threshold,set_pulse_width,sub_base=True,vmax=1000,plot_cond=None):
    cond_all = np.zeros_like(need_read,dtype=float)
    final_mask = np.ones_like(need_read,dtype=bool)
    tg_cond = {}
    for i in range(write_times):
        print(f"write_time = {i}")
        tg = start_tg+i*delta_tg
        chip.write4(crossbar=need_read,row_index=None,col_index=None,write_voltage=write_voltage,tg=tg,pulse_width=set_pulse_width,set_device=True,split_type=0,row_type=0,col_type=0)
        
        _,cond,_ = chip.read4(crossbar=need_read,row_index=None,col_index=None,read_voltage=0.1,tg=5,gain=1,sub_base=sub_base,from_row=True,split_type=0,row_type=0,col_type=0)
        np.save(f'../data/good_device/tg_cond2/tg_{tg}_cond.npy', cond[need_read])
        if i<=9:
            input_mask = need_read & (cond>50) & (cond<200)
        elif i==10:
            input_mask = need_read & (cond>50) & (cond<250)
        elif i==11:
            input_mask = need_read & (cond>50) & (cond<300)
        elif i==12 or i==13:
            input_mask = need_read & (cond>100) & (cond<400)
        elif i==14 or i==15:
            input_mask = need_read & (cond>200) & (cond<500)
        elif i==16 or i==17:
            input_mask = need_read & (cond>200) & (cond<600)
        elif i==18 or i==19:
            input_mask = need_read & (cond>200) & (cond<700)
        elif i==20:
            input_mask = need_read & (cond>300) & (cond<750)
        fit_mu, cond_mask = create_tg_cond_mask(cond, 50, tg, (need_read & (cond<1000)), input_mask)
        tg_cond[tg] = fit_mu
        final_mask = final_mask & cond_mask
        print(f"# of devices in range of +-50: {np.sum(final_mask)}")
        # if plot_cond: plot_cond(cond,title=f"tg={tg:.2f}-GoodDevices={np.sum(final_mask)}",vmax=vmax)
    return final_mask, tg_cond

In [ ]:
# crossbar = np.ones((256,256),dtype=bool)
crossbar = np.load("../data/good_device/20250621_cond_reset_200_set_500.npy")
final_mask, tg_cond = Set(chip=chip,need_read=crossbar,write_times=21,write_voltage=3,start_tg=0.8,delta_tg=0.05,threshold=600,set_pulse_width=1e-6,sub_base=True,vmax=1000,plot_cond=plot_cond)

In [ ]:
# with printoptions(threshold=np.inf):
#     print(need_read)
print(f"# of good devices = {np.sum(final_mask)}")
print(tg_cond)
linear_good_device_file_name = "../data/good_device/20250621_linear_tg_device_threshold_30.npy"
np.save(linear_good_device_file_name,final_mask)
with open("../data/good_device/tg_cond.pkl", "wb") as f:
    pickle.dump(tg_cond, f, protocol=pickle.HIGHEST_PROTOCOL)

### 2. Set操作

In [ ]:
need_read = np.ones((256,256),dtype=bool)
# need_read[:20,120:180]=True
select.Set(chip=chip,need_read=need_read,write_times=1,write_voltage=5,start_tg=1.5,delta_tg=0.05,threshold=800,set_pulse_width=100e-6,
           sub_base=True,vmax=1000,plot_cond=plot_cond)

In [ ]:
crossbar = np.ones((256,256))
_,cond,_ = chip.read4(crossbar=crossbar,row_index=None,col_index=None,read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=True,split_type=0,row_type=0,col_type=0)
plot_cond(cond)

In [ ]:
print(np.sum(cond>500))
set_good_device_file_name = "../data/good_device/20250613_cond_500_r8.npy"
np.save(set_good_device_file_name,cond>500)

In [ ]:
set_good_device_file_name = "../data/good_device/20250613_cond_500_r8.npy"
reset_good_device_file_name = "../data/good_device/20250613_cond_200_r8.npy"
cond_200 = np.load(reset_good_device_file_name)
cond_500 = np.load(set_good_device_file_name)
good_cond = cond_200&cond_500
overall_good_device_file_name = "../data/good_device/20250613_cond_reset_200_set_500.npy"
print(f"# of good devices = {np.sum(good_cond)}")
np.save(overall_good_device_file_name,good_cond)

### 6.22

In [ ]:
with open("../data/good_device/tg_cond.pkl", "rb") as file:
    tg_cond_dict = pickle.load(file)
tg_map = np.array(list(tg_cond_dict.keys()))
cond_map = np.array(list(tg_cond_dict.values()))
slope,intercept = np.polyfit(cond_map,tg_map, 1)

In [ ]:
# 6.15测试
min_G, max_G = 0, 1000
states = [min_G+i*((max_G-min_G)/17) for i in range(1,17)]
# print(", ".join([f"{i:.2f}" for i in states]))
threshold = 30

set_v,reset_v = 1.5,1
reset_pulse_width,set_pulse_width = 10e-6,100e-6
# need_read = np.ones((256,256),dtype=bool)
need_read = np.load("../data/good_device/20250621_linear_tg_device_threshold_30.npy")
need_read = need_read.astype(int)
target = np.ones((256,256))

for i in range(16):
    target[:, 0+i*16:16+i*16] *= states[i]
tg_v = target*slope+intercept  # -0.2
num_cycles = 100

In [ ]:
for k in range(1):
    for i in range(num_cycles):
        voltage_base = chip.read_point2(crossbar=need_read,read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
        voltage = chip.read_point2(crossbar=need_read,read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
        # resistence = chip.voltage_to_resistance(voltage = voltage-voltage_base)
        # cond = chip.compensation.compensation_point(resistence=resistence,from_row=True,return_type=0)
        cond = chip.voltage_to_cond(voltage=voltage-voltage_base)

        condition_reset = (cond > (target+threshold)) & need_read
        condition_set = (cond < (target-threshold)) & need_read
        error_cond = cond-target
        plot_cond(error_cond,vmin=-max_G,vmax=max_G,title=f"{i},reset:{np.sum(condition_reset)},set:{np.sum(condition_set)}")

        # if i>0:
        #     # set_pulse_width = set_pulse_width+10e-6*i
        #     # reset_pulse_width = reset_pulse_width+10e-6*i
        #     # set_v+=0.1
        #     # # reset_v+=0.05
        #     # tg_v[condition_set] +=0.025
        #     if i<10:
        #         tg_v[condition_reset] -= 0.04
        #         tg_v[condition_set] += 0.04
        #     else:
        #         tg_v[condition_reset] -= 0.02
        #         tg_v[condition_set] += 0.02

        # set_v += 0.05
        # tg_v[condition_set] += 0.02

        # tg_v.clip(0,3,out=tg_v)
        # reset
        chip.write_point2(crossbar=condition_reset,write_voltage=reset_v,tg=5,pulse_width=reset_pulse_width,set_device=False)
        chip.write_point2(crossbar=condition_reset,write_voltage=set_v-0.5,tg=tg_v,pulse_width=set_pulse_width,set_device=True)

        # set
        # # # chip.ps.set_time_out(100)
        chip.write_point2(crossbar=condition_set,write_voltage=set_v,tg=tg_v,pulse_width=set_pulse_width,set_device=True)
        # chip.write_point2(crossbar=condition_set,write_voltage=reset_v-0.5,tg=5,pulse_width=reset_pulse_width,set_device=False)

    # set_v = set_v+0.1
    # reset_v = reset_v+0.1
    # set_pulse_width = set_pulse_width+10e-6*k
    # reset_pulse_width = reset_pulse_width+10e-6*k
    # tg_v = target*slope+intercept+k*0.05-0.3


In [ ]:
# 误差直方图
voltage_base = chip.read_point2(crossbar=need_read,read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
voltage = chip.read_point2(crossbar=need_read,read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
cond = chip.voltage_to_cond(voltage=voltage-voltage_base)
error_cond = cond-target
selected_error_cond = error_cond[need_read.astype(bool)]
print(need_read.shape, np.sum(need_read))
print(selected_error_cond.shape)
# selected_error_cond = selected_error_cond.flatten()
mean = np.mean(selected_error_cond)
var = np.var(selected_error_cond) 
plt.hist(selected_error_cond, bins=50, density=True, alpha=0.6, color='b')
plt.title(f"Mean: {mean:.2f}, Variance: {var:.2f}")
plt.xlabel("Write error (uS)")
plt.ylabel("Frequency")
plt.show()